# Uncertainty Tests

This notebook prepares the calculation tables used by the analytical-uncertainty main-text figure and the Kd-bin supplementary figure. Plotting is kept in the figure notebooks/scripts.


# Setup and test data

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import sys

import numpy as np
import pandas as pd


def find_project_root(start=None):
    # Keep the notebook runnable from either the repository root or paper/notebooks.
    path = Path.cwd() if start is None else Path(start)
    for candidate in (path, *path.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Project root not found")


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_PATH = PROJECT_ROOT / "paper" / "data" / "independent_data_final.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "paper" / ".cache" / "uncertainty_tests"
ANALYTICAL_DIR = OUTPUT_DIR / "analytical"
KD_DIR = OUTPUT_DIR / "kd"
ANALYTICAL_DIR.mkdir(parents=True, exist_ok=True)
KD_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
data = pd.read_excel(DATA_PATH, sheet_name="Sheet1")
test_df = data.loc[data["training/testing"].str.lower().isin(["testing", "test"])].copy()
test_df["split"] = "test_subset"
test_df = test_df.reset_index(drop=True)

test_df.shape


# Analytical uncertainty

Each oxide is changed one at a time by its assigned relative error. The reported effect is half the difference between the +sigma and -sigma predictions. Rows with a zero baseline oxide are retained in the long table but excluded from the figure summaries.


In [ ]:
from aims4pt.model_tools.model_registry import get_models_initial_pools, import_all_models
from paper.scripts.constants_illustration import get_model_abbreviation


@dataclass(frozen=True)
class ModelSpec:
    target: str
    model: object
    name: str
    model_type: str


def get_model_specs(model_type):
    import_all_models()
    models = {}
    for target in ("P", "T"):
        for model in get_models_initial_pools(target, model_type, False):
            if model.cpx_only != (model_type == "cpx_only"):
                continue
            name = get_model_abbreviation(model.model_name, target)
            if target == "T" and name == "Pu08_32d" and "eq32a_P" not in model.model_name:
                continue
            key = (target, name)
            if key not in models:
                models[key] = ModelSpec(target, model, name, model_type)
    return list(models.values())


def split_inputs(df):
    cpx = df.filter(regex="_cpx$").apply(pd.to_numeric, errors="coerce").copy()
    liq = df.filter(regex="_liq$").apply(pd.to_numeric, errors="coerce").copy()
    cpx["P_kbar"] = pd.to_numeric(df["P (kbar)"], errors="coerce")
    cpx["T_C"] = pd.to_numeric(df["T (C)"], errors="coerce")
    return cpx, liq


def predict_model(spec, cpx, liq):

    values = spec.model.predict(cpx, liq)
    return pd.Series(np.asarray(values).reshape(-1), index=cpx.index, dtype=float)


cpx_liq_models = get_model_specs("cpx_liq")
cpx_only_models = get_model_specs("cpx_only")


In [ ]:
CPX_TOTAL_RANGE = (97.0, 103.0)
CPX_STOICH_RANGE = (0.9, 1.1)
RELATIVE_ERRORS = {
    "SiO2": 0.03,
    "TiO2": 0.08,
    "Al2O3": 0.03,
    "FeO": 0.03,
    "MgO": 0.03,
    "MnO": 0.08,
    "CaO": 0.03,
    "Na2O": 0.08,
    "Cr2O3": 0.08,
    "K2O": 0.08,
}
CPX_OXIDE_ORDER = ["SiO2", "TiO2", "Al2O3", "FeO", "MgO", "MnO", "CaO", "Na2O", "Cr2O3", "K2O"]
LIQ_OXIDE_ORDER = ["SiO2", "TiO2", "Al2O3", "FeO", "MgO", "MnO", "CaO", "Na2O", "K2O", "H2O"]


def oxide_label(base, phase):
    return f"{'FeOt' if base == 'FeO' else base}_{phase}"


def perturbation_specs(df, model_type):
    rows = []
    phases = ("cpx", "liq") if model_type == "cpx_liq" else ("cpx",)
    for phase in phases:
        order = CPX_OXIDE_ORDER if phase == "cpx" else LIQ_OXIDE_ORDER
        for base in order:
            column = f"{base}_{phase}"
            if column in df.columns:
                error = 0.20 if base == "H2O" else RELATIVE_ERRORS[base]
                rows.append((phase, column, oxide_label(base, phase), error))
    if model_type == "cpx_only" and "H2O_liq" in df.columns:
        rows.append(("liq", "H2O_liq", "H2O_liq", 0.20))
    return rows


def oxide_total(df, phase):
    columns = [
        column for column in df.columns
        if column.endswith(f"_{phase}") and not column.startswith("H2O")
    ]
    return df[columns].apply(pd.to_numeric, errors="coerce").sum(axis=1)


def cpx_stoich_ratio(df):
    si = df["SiO2_cpx"] / 60.0843
    ca = df["CaO_cpx"] / 56.0774
    mg = df["MgO_cpx"] / 40.3044
    fe = df["FeO_cpx"] / 71.844
    return (ca + fe + mg) / si


def qc_mask(df, phase):
    if phase == "cpx":
        total = oxide_total(df, "cpx")
        stoich = cpx_stoich_ratio(df)
        return total.between(*CPX_TOTAL_RANGE) & stoich.between(*CPX_STOICH_RANGE)
    return oxide_total(df, "liq") <= 105


def model_uses_oxide(spec, phase, column):
    base = column.removesuffix(f"_{phase}")
    aliases = ("FeO", "FeOt") if base == "FeO" else (base,)
    return any(
        str(name).endswith(f"_{phase}") and str(name).startswith(aliases)
        for name in (spec.model.standard_columns or [])
    )


In [ ]:
def run_analytical_uncertainty(df, model_specs):
    cpx_base, liq_base = split_inputs(df)
    baseline = {
        (spec.model_type, spec.target, spec.name): predict_model(spec, cpx_base, liq_base)
        for spec in model_specs
    }
    records = []

    for model_type in ("cpx_liq", "cpx_only"):
        selected_models = [spec for spec in model_specs if spec.model_type == model_type]
        for phase, column, display_name, rel_error in perturbation_specs(df, model_type):
            original = pd.to_numeric(df[column], errors="coerce").fillna(0.0)
            sigma = original * rel_error
            plus_df = df.copy()
            minus_df = df.copy()
            plus_df[column] = original + sigma
            minus_df[column] = original - sigma

            plus_ok = qc_mask(plus_df, phase)
            minus_ok = qc_mask(minus_df, phase)
            zero_value = original.eq(0)
            status = pd.Series("ok", index=df.index, dtype=object)
            status.loc[~plus_ok & minus_ok] = "qc_failed_plus"
            status.loc[plus_ok & ~minus_ok] = "qc_failed_minus"
            status.loc[~plus_ok & ~minus_ok] = "qc_failed_both"
            status.loc[zero_value] = "zero_value_skipped"
            ready = status.eq("ok")

            cpx_plus, liq_plus = split_inputs(plus_df)
            cpx_minus, liq_minus = split_inputs(minus_df)
            for spec in selected_models:
                base_prediction = baseline[(model_type, spec.target, spec.name)]
                plus_prediction = pd.Series(np.nan, index=df.index, dtype=float)
                minus_prediction = pd.Series(np.nan, index=df.index, dtype=float)
                not_used = not model_uses_oxide(spec, phase, column)

                if not_used:
                    plus_prediction.loc[ready] = base_prediction.loc[ready]
                    minus_prediction.loc[ready] = base_prediction.loc[ready]
                elif ready.any():
                    plus_prediction.loc[ready] = predict_model(
                        spec, cpx_plus.loc[ready], liq_plus.loc[ready]
                    )
                    minus_prediction.loc[ready] = predict_model(
                        spec, cpx_minus.loc[ready], liq_minus.loc[ready]
                        
                    )

                signed_effect = (plus_prediction - minus_prediction) / 2
                rows = pd.DataFrame(
                    {
                        "id": df["id"],
                        "model_type": model_type,
                        "model": spec.name,
                        "target_type": spec.target,
                        "phase": phase,
                        "oxide": display_name,
                        "rel_error": rel_error,
                        "baseline_value": original,
                        "signed_effect": signed_effect,
                        "abs_effect": signed_effect.abs(),
                        "status": status,
                    }
                )
                records.append(rows)

    return pd.concat(records, ignore_index=True)


In [ ]:
model_specs = cpx_liq_models + cpx_only_models
analytical_long = run_analytical_uncertainty(test_df, model_specs)

figure_rows = analytical_long.loc[
    analytical_long["status"].eq("ok")
    & analytical_long["baseline_value"].ne(0)
    & analytical_long["abs_effect"].notna()
].copy()


In [ ]:
analytical_group = [
    "model_type", "model", "target_type", "phase", "oxide", "rel_error",
]
analytical_summary = (
    figure_rows.groupby(analytical_group, dropna=False)
    .agg(
        n=("id", "nunique"),
        median_signed_effect=("signed_effect", "median"),
        median_abs_effect=("abs_effect", "median"),
        p95_abs_effect=("abs_effect", lambda values: values.quantile(0.95)),
        max_abs_effect=("abs_effect", "max"),
    )
    .reset_index()
)

count_scope = analytical_long.assign(
    nonzero_id=analytical_long["id"].where(analytical_long["baseline_value"].ne(0)),
    zero_id=analytical_long["id"].where(analytical_long["baseline_value"].eq(0)),
    plot_id=analytical_long["id"].where(
        analytical_long["status"].eq("ok")
        & analytical_long["baseline_value"].ne(0)
        & analytical_long["abs_effect"].notna()
    ),
)
count_group = ["model_type", "target_type", "phase", "oxide", "model"]
counts_by_model = (
    count_scope.groupby(count_group, dropna=False)
    .agg(
        total_sample_n=("id", "nunique"),
        nonzero_sample_n=("nonzero_id", "nunique"),
        zero_sample_n=("zero_id", "nunique"),
        plot_ok_n=("plot_id", "nunique"),
    )
    .reset_index()
)
counts_by_oxide = (
    counts_by_model.groupby(count_group[:-1], dropna=False)
    .agg(
        model_n=("model", "nunique"),
        total_sample_n=("total_sample_n", "max"),
        nonzero_sample_n=("nonzero_sample_n", "max"),
        zero_sample_n=("zero_sample_n", "max"),
        min_plot_ok_n=("plot_ok_n", "min"),
        median_plot_ok_n=("plot_ok_n", "median"),
        max_plot_ok_n=("plot_ok_n", "max"),
    )
    .reset_index()
)
counts_by_oxide["low_n_flag"] = np.where(
    counts_by_oxide["min_plot_ok_n"] < 10, "min_ok_n_lt_10", ""
)


In [ ]:
KEY_FEATURES = {
    # Cpx-only pressure
    ("cpx_only", "P", "Pet20"): {"Na2O_cpx", "Al2O3_cpx", "CaO_cpx"},
    ("cpx_only", "P", "Hig21"): {"Al2O3_cpx", "CaO_cpx", "Na2O_cpx", "MgO_cpx"},
    ("cpx_only", "P", "Jor22"): {"Al2O3_cpx", "Na2O_cpx", "CaO_cpx"},
    ("cpx_only", "P", "AgL24"): {"Al2O3_cpx", "Na2O_cpx", "CaO_cpx"},
    ("cpx_only", "P", "Wan21"): {"CaO_cpx", "FeOt_cpx", "Al2O3_cpx", "TiO2_cpx"},
    ("cpx_only", "P", "Chi23"): {"CaO_cpx", "SiO2_cpx", "Al2O3_cpx", "FeOt_cpx"},
    ("cpx_only", "P", "Pu08_32b"): {"H2O_liq"},
    # Cpx-liquid pressure
    ("cpx_liq", "P", "NP17"): {"Na2O_cpx", "CaO_cpx", "Al2O3_liq", "SiO2_liq", "K2O_liq"},
    ("cpx_liq", "P", "Pet20"): {"Na2O_cpx", "Al2O3_cpx", "MgO_liq", "CaO_cpx"},
    ("cpx_liq", "P", "Jor22"): {"Al2O3_cpx", "Na2O_cpx", "MgO_liq", "CaO_cpx"},
    ("cpx_liq", "P", "AgL24"): {"Al2O3_cpx", "MgO_liq", "Na2O_cpx", "CaO_cpx"},
    ("cpx_liq", "P", "Chi23"): {"K2O_liq", "CaO_cpx", "SiO2_cpx", "CaO_liq", "Al2O3_cpx", "Al2O3_liq"},
    # Cpx-only temperature
    ("cpx_only", "T", "Pet20"): {"CaO_cpx", "MgO_cpx", "MnO_cpx", "Al2O3_cpx", "Cr2O3_cpx"},
    ("cpx_only", "T", "Hig21"): {"MgO_cpx", "CaO_cpx", "MnO_cpx", "Al2O3_cpx"},
    ("cpx_only", "T", "Jor22"): {"CaO_cpx", "MgO_cpx", "Al2O3_cpx", "FeOt_cpx"},
    ("cpx_only", "T", "AgL24"): {"CaO_cpx", "MgO_cpx", "Al2O3_cpx", "FeOt_cpx"},
    ("cpx_only", "T", "Wan21"): {"FeOt_cpx", "H2O_liq", "CaO_cpx", "TiO2_cpx"},
    ("cpx_only", "T", "Chi23"): {"CaO_cpx", "SiO2_cpx", "Na2O_cpx", "Al2O3_cpx", "MgO_cpx"},
    # Cpx-liquid temperature
    ("cpx_liq", "T", "Pet20"): {"MgO_liq", "H2O_liq", "CaO_cpx", "Na2O_cpx", "CaO_liq", "Al2O3_cpx"},
    ("cpx_liq", "T", "Jor22"): {"MgO_liq", "CaO_cpx", "Na2O_cpx", "SiO2_liq", "Al2O3_cpx"},
    ("cpx_liq", "T", "AgL24"): {"MgO_liq", "CaO_cpx", "Al2O3_cpx", "Na2O_cpx", "Al2O3_liq"},
    ("cpx_liq", "T", "Chi23"): {"K2O_liq", "CaO_liq", "MgO_cpx", "Na2O_cpx", "FeOt_cpx", "MgO_liq"},
}

key_feature_rows = [
    {
        "model_type": model_type,
        "target_type": target,
        "model": model,
        "phase": "liq" if oxide.endswith("_liq") else "cpx",
        "oxide": oxide,
    }
    for (model_type, target, model), oxides in KEY_FEATURES.items()
    for oxide in sorted(oxides)
]
key_feature_summary = pd.DataFrame(key_feature_rows).merge(
    counts_by_model,
    on=["model_type", "target_type", "model", "phase", "oxide"],
    how="left",
).merge(
    analytical_summary[
        ["model_type", "target_type", "model", "phase", "oxide", "median_abs_effect", "p95_abs_effect", "max_abs_effect"]
    ],
    on=["model_type", "target_type", "model", "phase", "oxide"],
    how="left",
)


In [ ]:
analytical_long.to_csv(
    ANALYTICAL_DIR / "directional_equal_error_oat_long_test_subset.csv", index=False
)
analytical_summary.to_csv(
    ANALYTICAL_DIR / "directional_equal_error_oat_summary_test_subset.csv", index=False
)
counts_by_model.to_csv(
    ANALYTICAL_DIR / "directional_pm_abs_nonzero_oxide_counts_by_model_test_subset.csv", index=False
)
counts_by_oxide.to_csv(
    ANALYTICAL_DIR / "directional_pm_abs_nonzero_oxide_counts_by_oxide_test_subset.csv", index=False
)
key_feature_summary.to_csv(
    ANALYTICAL_DIR / "directional_key_feature_boxes_nonzero_oxide_counts_test_subset.csv", index=False
)

analytical_excel = ANALYTICAL_DIR / "analytical_uncertainty_tables.xlsx"
with pd.ExcelWriter(analytical_excel, engine="openpyxl") as writer:
    analytical_long.to_excel(writer, sheet_name="directional_long", index=False)
    analytical_summary.to_excel(writer, sheet_name="figure_summary", index=False)
    counts_by_model.to_excel(writer, sheet_name="counts_by_model", index=False)
    counts_by_oxide.to_excel(writer, sheet_name="counts_by_oxide", index=False)
    key_feature_summary.to_excel(writer, sheet_name="key_features", index=False)

analytical_summary.head()


# Kd uncertainty

Kd is calculated for the fixed clinopyroxene-liquid pairs. The supplementary figure uses signed pressure and temperature residuals summarized in 0.02-wide Kd bins; the vertical ranges are Q1-Q3.


In [ ]:
KD_BIN_EDGES = np.array([0.20, 0.22, 0.24, 0.26, 0.28, 0.30, 0.32, 0.34, 0.36])


def baseline_predictions(df, model_specs):
    cpx, liq = split_inputs(df)
    rows = []
    for spec in model_specs:
        prediction = predict_model(spec, cpx, liq)
        part = pd.DataFrame(
            {
                "id": df["id"],
                "split": "test_subset",
                "model": spec.name,
                "target_type": spec.target,
                "P_true": df["P (kbar)"],
                "T_true": df["T (C)"],
                "P_pred": prediction if spec.target == "P" else np.nan,
                "T_pred": prediction if spec.target == "T" else np.nan,
                "status": "ok",
            }
        )
        part["delta_P"] = part["P_pred"] - part["P_true"]
        part["delta_T"] = part["T_pred"] - part["T_true"]
        rows.append(part)
    return pd.concat(rows, ignore_index=True)


def calculate_kd(df):
    fe_mg_cpx = (df["FeO_cpx"] / 71.844) / (df["MgO_cpx"] / 40.304)
    fe_mg_liq = (df["FeO_liq"] / 71.844) / (df["MgO_liq"] / 40.304)
    kd = fe_mg_cpx / fe_mg_liq
    return pd.DataFrame(
        {
            "id": df["id"],
            "split": "test_subset",
            "Kd_FeMg": kd,
            "Kd_centered": kd - 0.28,
            "Kd_outside_0.20_0.36": ~kd.between(0.20, 0.36, inclusive="both"),
        }
    )


In [ ]:
baseline = baseline_predictions(test_df, cpx_liq_models)
kd_values = calculate_kd(test_df)
kd_merged = baseline.merge(kd_values[["id", "Kd_FeMg", "Kd_centered"]], on="id", how="left")


In [ ]:
def summarize_kd_bins(merged, bin_edges):
    edges = np.asarray(bin_edges, dtype=float)
    cut_edges = edges.copy()
    cut_edges[-1] += 1e-9
    labels = [f"{left:.2f}-{right:.2f}" for left, right in zip(edges[:-1], edges[1:])]
    centers = dict(zip(labels, (edges[:-1] + edges[1:]) / 2))

    samples = merged[["id", "Kd_FeMg"]].drop_duplicates()
    samples = samples.loc[samples["Kd_FeMg"].between(edges[0], edges[-1], inclusive="both")].copy()
    samples["Kd_bin"] = pd.cut(
        samples["Kd_FeMg"], bins=cut_edges, labels=labels, include_lowest=True, right=False
    )
    counts = pd.DataFrame(
        {
            "Kd_bin": labels,
            "Kd_bin_center": [centers[label] for label in labels],
            "sample_n": [
                samples.loc[samples["Kd_bin"] == label, "id"].nunique() for label in labels
            ],
        }
    )

    records = []
    for target in ("P", "T"):
        residual_column = f"delta_{target}"
        selected = merged.loc[
            merged["target_type"].eq(target)
            & merged["Kd_FeMg"].between(edges[0], edges[-1], inclusive="both")
            & merged[residual_column].notna()
        ].copy()
        selected["Kd_bin"] = pd.cut(
            selected["Kd_FeMg"], bins=cut_edges, labels=labels, include_lowest=True, right=False
        )
        for (model, kd_bin), group in selected.groupby(["model", "Kd_bin"], observed=True):
            residual = group[residual_column].astype(float)
            records.append(
                {
                    "model": model,
                    "target_type": target,
                    "value_metric": "signed_residual",
                    "Kd_bin": str(kd_bin),
                    "Kd_bin_center": centers[str(kd_bin)],
                    "n": len(residual),
                    "sample_n": group["id"].nunique(),
                    "median_residual": residual.median(),
                    "residual_q1": residual.quantile(0.25),
                    "residual_q3": residual.quantile(0.75),
                    "mean_residual": residual.mean(),
                    "rmse": np.sqrt(np.mean(residual**2)),
                }
            )
    return pd.DataFrame(records), counts


kd_bin_iqr, kd_bin_counts = summarize_kd_bins(kd_merged, KD_BIN_EDGES)


In [ ]:
baseline.to_csv(OUTPUT_DIR / "baseline_cpx_liq_predictions_test_subset.csv", index=False)
kd_values.to_csv(KD_DIR / "kd_values_by_sample_test_subset.csv", index=False)
kd_bin_iqr.to_csv(KD_DIR / "kd_bin_residual_iqr_0p02_by_model_test_subset.csv", index=False)
kd_bin_counts.to_csv(KD_DIR / "kd_bin_counts_0p02_test_subset.csv", index=False)

kd_excel = KD_DIR / "kd_uncertainty_tables.xlsx"
with pd.ExcelWriter(kd_excel, engine="openpyxl") as writer:
    baseline.to_excel(writer, sheet_name="baseline_predictions", index=False)
    kd_values.to_excel(writer, sheet_name="kd_by_sample", index=False)
    kd_bin_iqr.to_excel(writer, sheet_name="residual_iqr_0p02", index=False)
    kd_bin_counts.to_excel(writer, sheet_name="bin_counts_0p02", index=False)

pd.Series(
    {
        "test_samples": len(test_df),
        "analytical_rows": len(analytical_long),
        "valid_kd": kd_values["Kd_FeMg"].notna().sum(),
        "analytical_excel": analytical_excel,
        "kd_excel": kd_excel,
    }
)
